In [1]:
library(dplyr)
library(sccomp)
library(ggplot2)
library(forcats)
library(tidyr)
library(openxlsx)
library(cmdstanr)
library(ggplot2)
library(cowplot)
library(ggpubr)
library(rstatix)
library(tibble)
library(future)
library(loo)
set_cmdstan_path('/home/liyanguo/software/cmdstan-2.36.0')
plan(multisession)
mc.cores = 48


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


This is cmdstanr version 0.9.0

- CmdStanR documentation and vignettes: mc-stan.org/cmdstanr

- CmdStan path: /home/liyanguo/software/cmdstan-2.36.0

- CmdStan version: 2.36.0


Attaching package: ‘ggpubr’


The following object is masked from ‘package:cowplot’:

    get_legend



Attaching package: ‘rstatix’


The following object is masked from ‘package:stats’:

    filter


This is loo version 2.8.0

- Online documentation and vignettes at mc-stan.org/loo

- As of v2.0.0 loo defaults to 1 core but we recommend using as many as possible. Use the 'cores' argument or set options(mc.cores = NUM_CORES) for an entire session. 

CmdStan path set to: /home/liyanguo/software/cmdstan-2.36.0



# Age relate PBMC
#Exclude Neutrophil+basophils+platelet

## Classification_L4

### Data loading for sccomp

In [2]:
group = 'Ten_year_intervals'
classification = 'Classification_L4_PBMC'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
#system(paste0("rm ",path,'*'))

In [3]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [35]:
cell_types = c('HSC/MPP#','CILCP#','CLP#','MEP#','MkP#','MC/MCP#',
               
               'Naïve.B.cells#','Early.memory.B.cells#','Non-switched.memory.B.cells#','Switched.Memory.B.cells#',
               'CD95.memory.B.cells#',
               'Atypical.naïve.B.cells#','CD27+.IgD+.atypical.memory.B.cells#','CD27+.IgD-.atypical.memory.B.cells#','CD27-.IgD-.atypical.memory.B.cells#',
               'Plamsablasts#','IGKChi.Plasma.cells#','IGLL5hi.Plasma.cells#',
               
               'DN.T.cells#','Proliferative.DN.T.cells#',
               
               'Naïve.CD4+.T.cells#',
               'Tfh#','CD27+.Th1#','CD27-.Th1#','Th1/Th17#','CD27+.Th17#','CD27-.Th17#','Th2#','Th22#','Proliferative.help.memory.T.cells#',
               'GZMB+.CD4+.terminal.effector.T.cells#','CD4+.Temra#','HLA-DRhi.CD4+.terminal.effector.T.cells#','Proliferative.cytotoxic.CD4+.T.cells#',
               'Naïve.CD4+.Treg#','Memory.CD4+.Treg#','KLRB1+.CD4+.Treg#','HLA-DRhi.CD4+.Treg#','Proliferative.CD4+.Treg#',
               
               'Naïve.CD8+.T.cells#','CCR4+.CD8+.Tcm#','CCR4-.CD8+.Tcm#',
               'GZMK+.CD8+.Tem#','GZMB+.CD8+.Tem#','CD8+.Temra#','HLA-DRhi.CD8+.Tem#','Proliferative.CD8+.memory.T.cells#',
               'CD8+.Treg#',
               
               'CD27+.MAIT#','CD27-.MAIT#','CD56+.MAIT#',
               
               'Naïve.Vδ1+.T.cells#','CD279+.SOX4+.Vδ1+.T.cells#', 'SOX4+.Vδ1+.T.cells#','KLRC2+.effector.Vδ1+.T.cells#','GZMK+.effector.Vδ1+.T.cells#',
               'GZMK+.Vδ2+.T.cells#','GZMB+.Vδ2+.T.cells#','CD62Lhi.GZMK+.Vδ2+.T.cells#','Proliferative.Vδ2+.T.cells#',

               'iNKT#','vNKT#',
               
               'ILCP#','ILC2#','CD56bright.NK.cells#','CD56dim.NK.cells#','Adaptive.NK.cells#','Proliferative.NK.cells#',
               
               'Core.classical.monocytes#','GBP1+.classical.monocytes#','ISG+.classical.monocytes#','Intermediate.monocytes#', 'Non-classical.monocytes#',
               'ASDC#','CD1C+.cDC2#','CD14+.cDC2#','cDC1#','pDCs#','LAMP3+.DC#',
               'MPO+.CD177-.iLDNs#','MPO-.CD177-.iLDNs#','CD177int.iLDNs#','MMP8+.CD177+.iLDNs#','MMP9+.CD177+.iLDNs#','Proliferative.iLDNs#',
               'MPO+.mLDNs#','MMP8+.CD177+.mLDNs#','Proliferative.mLDNs#'
)

In [36]:
#showd be 88 cell type, 排除platelets、NDNs和Bas, 但包含mast cell: 
length(cell_types)

[1] 88

In [37]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID, DonorID, Batch, Gender, Age_in_Years,
       Ten_year_intervals, Pregnancy, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID, DonorID, Batch, Gender, Age_in_Years, Ten_year_intervals, Pregnancy, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = Ten_year_intervals) %>%
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Circadian.rhythm = factor(Sampling.time, levels = c('Morning','Evening'))
    ) %>%
    filter(Pregnancy == 'NO', Sampling.time== 'Morning')#可选删除样本

In [38]:
head(Level_counts_frac_long)

ClinicalID,DonorID,Batch,Gender,Age_in_Years,Intervals,Pregnancy,Sampling.time,groupby,groupby_count,Circadian.rhythm
<chr>,<chr>,<chr>,<fct>,<dbl>,<fct>,<fct>,<chr>,<chr>,<int>,<fct>
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,HSC/MPP#,17,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,CILCP#,5,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,CLP#,10,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,MEP#,0,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,MkP#,16,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,MC/MCP#,5,Morning


### Best model and plot

In [8]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [9]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Intervals + Gender, # + (1 | DonorID), # Pregnancy + Circadian.rhythm
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    ) %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [10]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [11]:
p1 = sccomp_boxplot(sccomp_result, factor = "Intervals",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Intervals)`
Warning message in stat_summary(aes(!!as.symbol(factor_of_interest), (generated_proportions)), :
“Ignoring unknown parameters: `outlier.shape`, `outlier.colour`, and
`outlier.size`”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Naïve.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Proliferative.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'SOX4+.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'KLRC2+.effector.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.gr

In [12]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD279+.SOX4+.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'SOX4+.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'KLRC2+.effector.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMK+.effector.Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD62Lhi.GZMK+.Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Naïve.Vδ1+.T.cells#' in 'mbcsToSbcs': for 

In [13]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [14]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ASDC#,(Intercept),NA,-1.96261855,-1.80150435,-1.63147188,0.0000,0.0000000,1.0008988,513.321,1759.525,-11.42144,-10.79926,-10.52768,0,0,1.003252,54.31138,47.1203
ASDC#,Intervals10-19,Intervals,-0.27062544,-0.05144896,0.15874820,0.9840,0.5558861,1.0040842,1462.892,1703.534,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals20-29,Intervals,-0.25459751,-0.03760269,0.15885709,0.9910,0.3449444,1.0010038,2119.085,1525.618,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals30-39,Intervals,-0.30317921,-0.10208979,0.08466951,0.9730,0.2434416,1.0006407,1931.927,1404.420,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals40-49,Intervals,-0.08104912,0.11416485,0.30488894,0.9720,0.2021933,1.0002517,1685.307,1611.545,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals50-59,Intervals,-0.36740854,-0.16410055,0.05143695,0.9115,0.1891104,0.9996177,2105.222,1965.485,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + intervals + FDR

In [13]:
group_col= 'Ten_year_intervals'
group_col_alias = 'Intervals'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')
baseline_group = '2-9'
sig_gsub = '%_in_PBMC'

In [14]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [15]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )

In [18]:
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col_alias)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 21, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text = element_text(size = 10, colour = 'black'),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [39]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [40]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('GenderMale', '(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub('Intervals', '', parameter),
           parameter = factor(parameter, levels = levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [41]:
#showd be 79 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 88

In [42]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )
ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=16)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Proliferative Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'CD62Lhi GZMK+ Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMB+ Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMK+ Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'GZMK+ effector Vδ1+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'KLRC2+ effector Vδ1+ T cells' in 'mbcsToSbcs': for δ (U+

## Classification_L3

### Data loading for sccomp

In [23]:
group = 'Ten_year_intervals'
classification = 'Classification_L3_PBMC'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
#system(paste0("rm ",path,'*'))

In [24]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L3_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [25]:
colnames(Level_counts_frac)[grepl('#',colnames(Level_counts_frac))]

[1] "ASDC#"                    "Atypical.B.cells#"       
 [3] "CD4+.Treg#"               "CD8+.Tcm#"               
 [5] "CD8+.Tem#"                "CD8+.Treg#"              
 [7] "Classical.monocytes#"     "Cytotoxic.CD4+.T.cells#" 
 [9] "DN.T.cells#"              "HSPC#"                   
[11] "Help.memory.T.cells#"     "Intermediate.monocytes#" 
[13] "LAMP3+.DC#"               "LDNs#"                   
[15] "MAIT#"                    "Mast.cells#"             
[17] "Memory.B.cells#"          "NK.cells#"               
[19] "NKT#"                     "Naïve.B.cells#"          
[21] "Naïve.CD4+.T.cells#"      "Naïve.CD8+.T.cells#"     
[23] "Non-NK.ILCs#"             "Non-classical.Monocytes#"
[25] "Plasma.cells#"            "Vδ1+.T.cells#"           
[27] "Vδ2+.T.cells#"            "cDC1#"                   
[29] "cDC2#"                    "pDCs#"                   
[31] "Basophils#"               "NDNs#"                   
[33] "NEUT#"                    "LYMPH#"                  
[35] "MONO#"                    "EOS#"                    
[37] "BASO#"

In [26]:
cell_types = c('ASDC#','LAMP3+.DC#', 'cDC1#','cDC2#','pDCs#',
               'Naïve.B.cells#','Atypical.B.cells#','Memory.B.cells#','Plasma.cells#',
               'DN.T.cells#',
               'Naïve.CD4+.T.cells#',
               'Help.memory.T.cells#',
               'Cytotoxic.CD4+.T.cells#',
               'CD4+.Treg#','CD8+.Treg#',
               'Naïve.CD8+.T.cells#','CD8+.Tcm#','CD8+.Tem#',
               'MAIT#','NKT#',
               'Vδ1+.T.cells#','Vδ2+.T.cells#',
               'NK.cells#','Non-NK.ILCs#',
               
               'HSPC#','Mast.cells#',
               
               'Classical.monocytes#', 'Intermediate.monocytes#','Non-classical.Monocytes#',
               'LDNs#'
)

In [27]:
#showd be 30 cell type, 排除platelets、NDNs和Bas, 但包含mast cell
length(cell_types)

[1] 30

In [28]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID, DonorID, Batch, Gender, Age_in_Years,
       Ten_year_intervals, Pregnancy, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID, DonorID, Batch, Gender, Age_in_Years, Ten_year_intervals, Pregnancy, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = Ten_year_intervals) %>%
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Circadian.rhythm = factor(Sampling.time, levels = c('Morning','Evening'))
    ) %>%
    filter(Pregnancy == 'NO', Sampling.time== 'Morning')#可选删除样本

In [29]:
head(Level_counts_frac_long)

ClinicalID,DonorID,Batch,Gender,Age_in_Years,Intervals,Pregnancy,Sampling.time,groupby,groupby_count,Circadian.rhythm
<chr>,<chr>,<chr>,<fct>,<dbl>,<fct>,<fct>,<chr>,<chr>,<int>,<fct>
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,ASDC#,6,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,LAMP3+.DC#,0,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,cDC1#,24,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,cDC2#,391,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,pDCs#,191,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,Naïve.B.cells#,1617,Morning


### Best model and plot

In [30]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [31]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Intervals + Gender,# + (1 | DonorID), # Pregnancy + Circadian.rhythm
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    ) %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [32]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [33]:
p1 = sccomp_boxplot(sccomp_result, factor = "Intervals",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Intervals)`
Warning message in stat_summary(aes(!!as.symbol(factor_of_interest), (generated_proportions)), :
“Ignoring unknown parameters: `outlier.shape`, `outlier.colour`, and
`outlier.size`”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”


In [34]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+.T.cells#' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsA

In [35]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [36]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ASDC#,(Intercept),NA,-3.54780684,-3.389264190,-3.2375932,0.0000,0.0000000,1.0042494,424.9076,893.2568,-14.35584,-11.01112,-10.25262,0,0,1.00477,63.7672,58.40945
ASDC#,Intervals10-19,Intervals,-0.19597146,-0.009084942,0.1924139,0.9770,0.4276071,0.9997056,1164.8944,1794.9524,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals20-29,Intervals,-0.14755311,0.034490076,0.2225181,0.9540,0.2486250,1.0006209,1266.6451,1904.6839,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals30-39,Intervals,-0.19210054,-0.020409455,0.1524330,0.9815,0.2319000,1.0003324,1585.8597,1856.0228,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals40-49,Intervals,-0.03493659,0.146085234,0.3340715,0.7180,0.1693519,0.9998361,1451.1591,1550.4050,NA,NA,NA,NA,NA,NA,NA,NA
ASDC#,Intervals50-59,Intervals,-0.28229974,-0.074533685,0.1276312,0.8920,0.2333276,1.0001955,1259.8376,1847.1500,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + intervals + FDR

In [37]:
group_col= 'Ten_year_intervals'
group_col_alias = 'Intervals'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')
baseline_group = '2-9'
sig_gsub = '%_in_PBMC'

In [38]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_"), 
    contains("v_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [39]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col_alias)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 21, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text = element_text(size = 10, colour = 'black'),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        group1 = "group1",
        group2 = "group2",
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存图表（文件名适配细胞类型和分组）
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”


### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [40]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [41]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('GenderMale', '(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub('Intervals', '', parameter),
           parameter = factor(parameter, levels = levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [42]:
#showd be 29 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 30

In [43]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )

ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=16)

Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ2+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”
Warning message in grid.Call.graphics(C_text, as.graphicsAnnot(x$label), x$x, x$y, :
“conversion failure on 'Vδ1+ T cells' in 'mbcsToSbcs': for δ (U+03B4)”


# Age relate NDNs
#Exclude PBMC+platelet, basophils also not mention

## Classification_L4

### Data loading for sccomp

In [26]:
group = 'Ten_year_intervals'
classification = 'Classification_L4_NDNs'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
#system(paste0("rm ",path,'*'))

In [27]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [38]:
# cell_types = c('Core.NDNs#',
#                'MT-ATP6-.MT-CO2-.NDNs#',
# 'IRF1-.GBP2-.NDNs#',
# 'VIM-.FLNA-.NDNs#',
#                'ALPL-.MARCKS-.NDNs#',
#                'RGS2-.NDNs#',
#                'IFIT2-.RNF213-.NDNs#',
#                'CXCL8-.PTGS2+.NDNs#',
#                'FOS-.NDNs#'
# )
cell_types = c('Core.NDNs#','IRF1-.GBP2-.NDNs#',
                'IFIT2-.RNF213-.NDNs#',
               'VIM-.FLNA-.NDNs#',
               'CXCL8-.PTGS2+.NDNs#',
                'FOS-.NDNs#', 
               'MT-ATP6-.MT-CO2-.NDNs#',
               'ALPL-.MARCKS-.NDNs#',
               'RGS2-.NDNs#'
                
)

In [29]:
#showd be 9 cell type, 排除platelets、PBMC、iLDNs和Bas
length(cell_types)

[1] 9

In [30]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID, DonorID, Batch, Gender, Age_in_Years,
       Ten_year_intervals, Pregnancy, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID, DonorID, Batch, Gender, Age_in_Years, Ten_year_intervals, Pregnancy, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = Ten_year_intervals) %>%
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Circadian.rhythm = factor(Sampling.time, levels = c('Morning','Evening'))
    ) %>%
    filter(Pregnancy == 'NO', Sampling.time== 'Morning')#可选删除样本

In [31]:
head(Level_counts_frac_long)

ClinicalID,DonorID,Batch,Gender,Age_in_Years,Intervals,Pregnancy,Sampling.time,groupby,groupby_count,Circadian.rhythm
<chr>,<chr>,<chr>,<fct>,<dbl>,<fct>,<fct>,<chr>,<chr>,<int>,<fct>
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,Core.NDNs#,5119,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,IRF1-.GBP2-.NDNs#,3031,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,VIM-.FLNA-.NDNs#,2036,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,CXCL8-.PTGS2+.NDNs#,1633,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,IFIT2-.RNF213-.NDNs#,1076,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,ALPL-.MARCKS-.NDNs#,1050,Morning


### Best model and plot

In [50]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [51]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Intervals + Gender, # Pregnancy + Circadian.rhythm
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    )  %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [52]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [53]:
sccomp_result = readRDS(paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [54]:
p1 = sccomp_boxplot(sccomp_result, factor = "Intervals",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Intervals)`
Warning message in stat_summary(aes(!!as.symbol(factor_of_interest), (generated_proportions)), :
“Ignoring unknown parameters: `outlier.shape`, `outlier.colour`, and
`outlier.size`”


In [55]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

In [56]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [57]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
ALPL-.MARCKS-.NDNs#,(Intercept),NA,0.1476348,0.23356709,0.32558435,0.2265,0.03783333,0.9998237,1554.447,1744.086,-5.082325,-5.021911,-4.955834,0,0,0.9999929,1545.117,1487.034
ALPL-.MARCKS-.NDNs#,Intervals10-19,Intervals,-0.1028800,0.01935308,0.13647040,0.9980,0.54793750,1.0002164,1775.550,1818.608,NA,NA,NA,NA,NA,NA,NA,NA
ALPL-.MARCKS-.NDNs#,Intervals20-29,Intervals,-0.2873932,-0.16944613,-0.04525486,0.7075,0.35233333,1.0004979,1690.303,1848.879,NA,NA,NA,NA,NA,NA,NA,NA
ALPL-.MARCKS-.NDNs#,Intervals30-39,Intervals,-0.2906137,-0.18529131,-0.07690752,0.6045,0.16137500,0.9995021,1846.448,2000.484,NA,NA,NA,NA,NA,NA,NA,NA
ALPL-.MARCKS-.NDNs#,Intervals40-49,Intervals,-0.3255116,-0.21343437,-0.10595451,0.3875,0.10140000,0.9995812,1733.138,1705.917,NA,NA,NA,NA,NA,NA,NA,NA
ALPL-.MARCKS-.NDNs#,Intervals50-59,Intervals,-0.2832946,-0.16667304,-0.04489859,0.7235,0.33387500,0.9999760,1586.524,1507.458,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + intervals + FDR

In [58]:
group_col= 'Ten_year_intervals'
group_col_alias = 'Intervals'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')
baseline_group = '2-9'
sig_gsub = '%_in_NEU_BAS'

In [59]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_"), 
    contains("v_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [60]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col_alias)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 21, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text = element_text(size = 10, colour = 'black'),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        group1 = "group1",
        group2 = "group2",
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存图表（文件名适配细胞类型和分组）
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”


### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [39]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [40]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('GenderMale', '(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub('Intervals', '', parameter),
           parameter = factor(parameter, levels = levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [41]:
#showd be 79 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 9

In [42]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )

ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=5)

# Age relate iLDNs

#Exclude PBMC+platelet, basophils also not mention

## Classification_L4

### Data loading for sccomp

In [65]:
group = 'Ten_year_intervals'
classification = 'Classification_L4_LDNs'
#output
path = paste0('/home/liyanguo/MyImmuCell/08_supp_outputs/sccomp_age/',classification,'/')
system(paste0("mkdir -p ",path))
#system(paste0("rm ",path,'*'))

In [66]:
Level_counts_frac = read.xlsx("/home/liyanguo/MyImmuCell/06_Finnal_raw_count/Classification_L4_donor_counts_frac_clinicl_table.xlsx",
                              rowNames=T)

In [67]:
cell_types = c('MPO+.CD177-.iLDNs#','MPO-.CD177-.iLDNs#','CD177int.iLDNs#','MMP8+.CD177+.iLDNs#','MMP9+.CD177+.iLDNs#','Proliferative.iLDNs#',
               'MPO+.mLDNs#','MMP8+.CD177+.mLDNs#','Proliferative.mLDNs#'
)

In [68]:
#showd be 9 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(cell_types)

[1] 9

In [69]:
Level_counts_frac_long = Level_counts_frac %>%
    select(ClinicalID, DonorID, Batch, Gender, Age_in_Years,
       Ten_year_intervals, Pregnancy, Sampling.time, all_of(cell_types)) %>%
    pivot_longer(
      cols = -c(ClinicalID, DonorID, Batch, Gender, Age_in_Years, Ten_year_intervals, Pregnancy, Sampling.time),
      names_to = "groupby",
      values_to = "groupby_count"
    ) %>%
    rename(Intervals = Ten_year_intervals) %>%
    mutate(
      groupby_count = as.integer(groupby_count),
      Intervals = factor(Intervals, levels = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')),
      Gender = factor(Gender, levels = c('Female','Male')),
      Pregnancy = factor(Pregnancy, levels = c('NO','YES')),
      Circadian.rhythm = factor(Sampling.time, levels = c('Morning','Evening'))
    ) %>%
    filter(Pregnancy == 'NO', Sampling.time== 'Morning')

In [70]:
head(Level_counts_frac_long)

ClinicalID,DonorID,Batch,Gender,Age_in_Years,Intervals,Pregnancy,Sampling.time,groupby,groupby_count,Circadian.rhythm
<chr>,<chr>,<chr>,<fct>,<dbl>,<fct>,<fct>,<chr>,<chr>,<int>,<fct>
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,MPO+.CD177-.iLDNs#,23,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,MPO-.CD177-.iLDNs#,29,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,CD177int.iLDNs#,37,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,MMP8+.CD177+.iLDNs#,43,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,MMP9+.CD177+.iLDNs#,17,Morning
D0009,D0009,B4,Male,26.8,20-29,NO,Morning,Proliferative.iLDNs#,15,Morning


### Best model and plot

In [71]:
#年龄作为fixed effect
#性别作为fixed effect，但效果一般。已有研究表明性别对于免疫细胞比例影响局限在NK和单核细胞 (Effects of sex and aging on the immune cell landscape as assessed by single-cell transcriptomic analysis)
#仅考虑个体间DonorID的差异和实验批次Batch的差异,使用随机截距Intercept, 旨在捕捉个体和批次基线差异

In [72]:
sccomp_result = Level_counts_frac_long %>%
    sccomp_estimate(
    formula_composition = ~ Intervals + Gender, # Pregnancy + Circadian.rhythm
    sample = "ClinicalID",
    cell_group = "groupby",
    abundance = "groupby_count",
    bimodal_mean_variability_association = TRUE,
    cores = 48, verbose = FALSE
    )  %>% sccomp_remove_outliers(cores = 48, verbose = FALSE, max_sampling_iterations = 2000) %>% sccomp_test(test_composition_above_logit_fold_change = 0.2)

sccomp says: groupby_count column is an integer. The sum-constrained beta binomial model will be used

sccomp says: estimation

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...

sccomp says: to do hypothesis testing run `sccomp_test()`,
  the `test_composition_above_logit_fold_change` = 0.1 equates to a change of ~10%, and
  0.7 equates to ~100% increase, if the baseline is ~0.1 proportion.
  Use `sccomp_proportional_fold_change` to convert c_effect (linear) to proportion difference (non-linear).

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier identification - step 1/2

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 48 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


sccomp says: outlier-free model fitting - step 2/2

sccomp says: the composition design matrix has columns: (Intercept), Intervals10-19, Intervals20-29, Intervals30-39, Intervals40-49, Intervals50-59, Intervals60-69, Intervals70-79, Intervals80-89, Intervals90+, GenderMale

sccomp says: the variability design matrix has columns: (Intercept)

Loading model from cache...



In [73]:
saveRDS(sccomp_result,paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [74]:
sccomp_result = readRDS(paste0(dirname(path),'/',classification,"sccomp_result.RDS"))

In [75]:
p1 = sccomp_boxplot(sccomp_result, factor = "Intervals",)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_boxplot.pdf"),
        p1,width=80,height=20,limitsize = FALSE)

sccomp says: When visualising proportions, especially for complex models, consider setting `remove_unwanted_effects=TRUE`. This will adjust the proportions, preserving only the observed effect.

Loading model from cache...



Running standalone generated quantities after 1 MCMC chain, with 1 thread(s) per chain...

Chain 1 finished in 0.0 seconds.


Joining with `by = join_by(groupby, ClinicalID)`
Joining with `by = join_by(groupby, Intervals)`
Warning message in stat_summary(aes(!!as.symbol(factor_of_interest), (generated_proportions)), :
“Ignoring unknown parameters: `outlier.shape`, `outlier.colour`, and
`outlier.size`”
Warning message:
“Removed 18 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 18 rows containing missing values or values outside the scale range
(`geom_point()`).”


In [76]:
p2 = plot_1D_intervals(sccomp_result[!grepl("___D", sccomp_result$parameter),])
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_1D.pdf"),
        p2,width=20,height=20,limitsize = FALSE)

In [77]:
p3 = plot_2D_intervals(sccomp_result)
ggsave2(paste0(dirname(path),'/',classification, "sccomp_plot_2D.pdf"),
        p3,width=20,height=20,limitsize = FALSE)

In [78]:
head(sccomp_result)

groupby,parameter,factor,c_lower,c_effect,c_upper,c_pH0,c_FDR,c_rhat,c_ess_bulk,c_ess_tail,v_lower,v_effect,v_upper,v_pH0,v_FDR,v_rhat,v_ess_bulk,v_ess_tail
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
CD177int.iLDNs#,(Intercept),NA,0.12977998,0.30455524,0.48509588,0.1230,0.0220000,1.0007150,1801.564,1514.994,-5.6008,-5.303816,-4.971405,0,0,1.000168,349.792,517.4609
CD177int.iLDNs#,Intervals10-19,Intervals,-0.02020845,0.23006664,0.46479531,0.3935,0.1498333,1.0008562,1805.239,1676.845,NA,NA,NA,NA,NA,NA,NA,NA
CD177int.iLDNs#,Intervals20-29,Intervals,-0.31524110,-0.09628611,0.11578165,0.8350,0.4368750,0.9997185,1950.020,1901.159,NA,NA,NA,NA,NA,NA,NA,NA
CD177int.iLDNs#,Intervals30-39,Intervals,-0.42232385,-0.22659849,-0.03539099,0.4040,0.2332500,0.9999954,1299.924,1919.918,NA,NA,NA,NA,NA,NA,NA,NA
CD177int.iLDNs#,Intervals40-49,Intervals,-0.26106645,-0.05926735,0.14248174,0.9140,0.4835000,1.0002240,1664.417,1864.370,NA,NA,NA,NA,NA,NA,NA,NA
CD177int.iLDNs#,Intervals50-59,Intervals,-0.32717302,-0.10404555,0.08805600,0.8115,0.4740714,0.9996173,1313.044,1892.783,NA,NA,NA,NA,NA,NA,NA,NA


### sccomp boxplot prop + intervals + FDR

In [79]:
group_col= 'Ten_year_intervals'
group_col_alias = 'Intervals'
color_pal <- c('#D2EBC8','#7DBFA7','#EE934E','#AECDE1','#8FA4AE','#BBDD78','#8AD9DF','#EBCC96','#9FD2A8','#4169E1')
levels_group = c('2-9','10-19','20-29','30-39','40-49','50-59','60-69','70-79','80-89','90+')
baseline_group = '2-9'
sig_gsub = '%_in_NEU_BAS'

In [80]:
sig_data = sccomp_result %>% 
    pivot_longer(c(contains("c_"), 
    contains("v_")), names_pattern = "([cv])_([a-zA-Z0-9]+)", 
    names_to = c("which", "stats_name"), values_to = "stats_value"
                ) %>% 
    filter(stats_name == "FDR",stats_value < 1,factor == factor
          ) %>% mutate(group2 = baseline_group
                      ) %>%  mutate(group1 = gsub(group_col_alias, '', parameter))

In [81]:
data_proportion = attr(sccomp_result,'count_data') %>% 
    select(ClinicalID,groupby,groupby_count,all_of(group_col_alias))  %>% 
    with_groups(
    .groups = ClinicalID,
    .f = ~ mutate(
      .x,
      proportion = groupby_count / sum(groupby_count, na.rm = TRUE) *100
    )
  )
for (cell_type in cell_types){
    
    plot_data <- data_proportion  %>% select(-groupby_count) %>% 
        filter(groupby == cell_type) %>% 
        rename(plot_group = !!group_col_alias)
    baseline_mean <- plot_data %>% 
          filter(plot_group == baseline_group) %>% 
          summarise(me = mean(proportion, na.rm = TRUE)) %>% 
          pull(me)
    p_values <- sig_data %>% 
          filter(factor ==group_col_alias,
                 groupby == cell_type) %>%
          add_significance(
            p.col = 'stats_value', 
            output.col = 'stars',
            cutpoints = c(0, 0.001, 0.01, 0.05, 1),
            symbols = c("***", "**", "*", "ns")
          ) %>% 
          filter(stars != 'ns')

    p <- ggplot(plot_data, aes(x = plot_group, y = proportion)) +
      geom_boxplot(aes(color = plot_group,fill = plot_group), width = 0.5, outlier.shape = NA, color = 'black') +
      geom_jitter(aes(color = plot_group,fill = plot_group), shape = 21, width = 0.05, size = 1.5, alpha = 0.5) +#, color = 'LightGrey',stroke = 0.1
      theme_cowplot() +
      theme(
        axis.text.y = element_text(size = 10),
        axis.text.x = element_text(angle = 45, hjust = 1),
        axis.title = element_text(size = 10, colour = 'black'),
        legend.position = 'none'
      ) +
      labs(
        title = paste0(baseline_group, " as baseline (effect >0.2 in 95% CI)"),
        subtitle = '***:<0.001, **:<0.01, *:<0.05',
        y = 'Percentage', x = group_col
      ) +
      geom_hline(yintercept = as.numeric(baseline_mean), linetype = 'dashed', alpha = 0.5) +
      stat_pvalue_manual(
        p_values, 
        group1 = "group1",
        group2 = "group2",
        y = max(plot_data$proportion, na.rm = TRUE) * 1.1,
        step.increase = 0.05,
        label = "{stars}",
        size = 5, 
        hide.ns = TRUE,
        tip.length = 0.01
      )+
      scale_color_manual(values = color_pal,aesthetics = c("colour", "fill"))
    
    # 保存图表（文件名适配细胞类型和分组）
    cell_type_B = gsub('[#]', sig_gsub, cell_type)
    ggsave2(paste0(path,
                   gsub('[.|%|/]','',cell_type_B),'_',group_col,".pdf"),
            p,width=8,heigh=8)
    }

Warning message:
“Removed 2 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_point()`).”
Warning message in (function (mapping = NULL, data = NULL, stat = "identity", position = "nudge", :
“Ignoring unknown parameters: `group1` and `group2`”
Warning message:
“Removed 2 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_point()`).”
Warning message:
“Removed 2 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_point()`).”
Warning message:
“Removed 2 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_point()`).”

### cell x intervals to vis the effect size and FDR
#参考组可视化effect没有意义

In [82]:
cell_types_clean <- cell_types %>%
  gsub('[#]', '', .) %>%
  gsub('[.]', ' ', .)

In [83]:
data  = sccomp_result %>% 
    select(groupby,parameter,c_effect,c_FDR) %>% 
    add_significance(p.col='c_FDR',
                     output.col='stars',
                     cutpoints = c(0, 0.001, 0.01, 0.05, 1),
                     symbols = c("***", "**", "*", ""))  %>% 
    filter(!parameter %in% c('GenderMale', '(Intercept)')) %>% 
    filter(!grepl("___D", parameter)) %>% 
    mutate(parameter = gsub('Intervals', '', parameter),
           parameter = factor(parameter, levels = levels_group),
           groupby = gsub('[#]', '', groupby),
           groupby = gsub('[.]', ' ', groupby),
           groupby = factor(groupby, levels = rev(cell_types_clean))
          )

In [84]:
#showd be 6 cell type, 排除platelets、中性粒和Bas, 但包含mast cell
length(table(data$groupby))

[1] 9

In [85]:
p <- ggplot(data, aes(x = parameter, y = groupby, fill = c_effect)) +
    geom_tile(color = "white", linewidth = 0.5) +
    geom_text(aes(label = stars), color = "black", size = 3) +
    scale_fill_gradient2(low = "#008B8B", mid = "white", high = "#A52A2A", midpoint = 0, name = "Effect size") +
    theme_classic() +
    theme(
      axis.text.x = element_text(angle = 45, hjust = 1, size = 10),
      axis.text.y = element_text(size = 10),
      axis.title = element_text(size = 12, face = "bold"),
      plot.title = element_text(size = 14, face = "bold", hjust = 0.5),
      legend.title = element_text(size = 10),
      legend.key.size = unit(0.8, "cm")
    ) +
    labs(
      title = "Robust differential composition (2-9 as baseline)",
      subtitle = '***:<0.001, **:<0.01, *:<0.05',
      x = "Age group", y = "Cell type"
    )

ggsave2(paste0(path,'Effect_size_',classification,".pdf"),p,width=8,heigh=5)